In [ ]:
def build_doodle_model(num_classes):
    # 1. Define our Grayscale Input
    img_input = layers.Input(shape=(128, 128, 1))
    
    # 2. Preprocessing: Scale [0, 255] to [-1, 1]
    x = layers.Rescaling(1./127.5, offset=-1)(img_input)
    
    # 3. Adapter: 1-to-3 Channel Conversion
    x = layers.Conv2D(3, (1, 1), padding='same')(x)
    
    # 4. Load Base Model WITHOUT passing the tensor yet
    # This avoids the layer count mismatch during weight loading
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(128, 128, 3), # Standard RGB shape
        include_top=False,
        weights='imagenet'
    )
    
    # 5. Connect our adapter (x) to the base_model
    x = base_model(x, training=False) 
    
    # 6. Classifier Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x) 
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(img_input, outputs)
    
    # NOTE: Since we used 'base_model(x)', the Stage 2 unfreezing 
    # needs to refer to base_model directly.
    return model, base_model

model, base_model = build_doodle_model(num_classes)